## 1. Import Libraries

In [ ]:
import os
import random
import warnings
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from skimage.feature import graycomatrix, graycoprops, hog, local_binary_pattern

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, matthews_corrcoef,
                             precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import ParameterGrid, StratifiedKFold, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
SEED = 42
os.makedirs("figures", exist_ok=True)


def show_fig(name):
    """Save the current figure to ./figures (for the report) and display it."""
    safe = "".join(c if c.isalnum() else "_" for c in name.lower()).strip("_")
    plt.savefig(f"figures/{safe}.png", dpi=200, bbox_inches="tight")
    plt.show()

np.random.seed(SEED)
random.seed(SEED)

print("OpenCV:", cv2.__version__)

### Deep-Learning Library
The neural-architecture section uses TensorFlow/Keras. If TensorFlow is unavailable, every section before Section 12 still runs.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D,
                                         Dense, Dropout)
    from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
    from tensorflow.keras.callbacks import EarlyStopping
    tf.random.set_seed(SEED)
    print("TensorFlow version:", tf.__version__)
except ImportError:
    tf = None
    print("TensorFlow is not installed. Section 12 will be skipped.")

### Workflow Overview

The diagram summarises the experimental design: one dataset, one preprocessing step, one stratified hold-out test set, and two model families.

In [ ]:
# Workflow overview (Figure 1 of the report)
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(10, 5.6))
ax.axis("off"); ax.set_xlim(0, 100); ax.set_ylim(0, 60)

def box(x, y, w, h, text, face="#EEF2F8", edge="#4C72B0"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.4,rounding_size=1.2",
                                fc=face, ec=edge, lw=1.2))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=9)

def arrow(x1, y1, x2, y2):
    ax.annotate("", (x2, y2), (x1, y1), arrowprops=dict(arrowstyle="-|>", color="#444", lw=1.1))

box(1, 24, 13, 12, "400 fundus\nimages\n(300 normal,\n100 cataract)")
box(18, 24, 15, 12, "Preprocessing\ncrop, 224x224,\nCLAHE (LAB L)")
box(37, 24, 13, 12, "Stratified\n80:20 split\n(seed 42)")
box(54, 38, 19, 14, "Classical branch\n401 features\nScaler, PCA 95%\nnone / weights /\nROS / SMOTE")
box(54, 6, 19, 14, "Neural branch\n20% validation split;\nnone / weights /\naugmentation", "#FBEFE5", "#DD8452")
box(77, 38, 21, 14, "SVM, Random Forest,\nLogistic Regression\n(12 experiments)")
box(77, 6, 21, 14, "Small CNN, MobileNetV2,\nEfficientNetB0\n(9 experiments)", "#FBEFE5", "#DD8452")
for a in [(14, 30, 18, 30), (33, 30, 37, 30), (50, 32, 54, 45), (50, 28, 54, 13), (73, 45, 77, 45), (73, 13, 77, 13)]:
    arrow(*a)
ax.text(50, 57, "Same 80-image hold-out test set: recall, specificity, precision, F1, MCC, ROC-AUC, average precision",
        ha="center", fontsize=9, style="italic", color="#333")
plt.tight_layout()
show_fig("figure1_workflow")

## 2. Dataset Configuration

Expected folder structure:

```text
retina_dataset/dataset/
├── 1_normal/        (300 images)
│   ├── NL_001.png
│   └── ...
└── 2_cataract/      (100 images)
    ├── cataract_001.png
    └── ...
```

The `3_` and `2_glaucoma` folders of the public repository are not used. Plain folder names (`normal`, `cataract`) are also accepted. Change `DATASET_PATH` if the dataset is stored elsewhere.

In [ ]:
# Folder that holds 1_normal and 2_cataract. A relative path works when the notebook
# sits next to the "dataset" folder; replace it with a full path otherwise.
DATASET_PATH = "dataset"
CATEGORIES = ["normal", "cataract"]
FOLDER_NAMES = {"normal": ["1_normal", "normal"], "cataract": ["2_cataract", "cataract"]}
LABEL_MAP = {"normal": 0, "cataract": 1}
IMG_SIZE = 224
SIMULATE_RATIO = None  # Keep original 3:1 class ratio in the actual experiment
EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")


print("Dataset path:", DATASET_PATH)
print("Categories  :", CATEGORIES)

## 3. Read File Names and Count the Images in Each Class

Class imbalance is measured before any processing, because the imbalance ratio determines which handling techniques are appropriate.

In [ ]:
records = []
for category in CATEGORIES:
    candidates = [os.path.join(DATASET_PATH, name) for name in FOLDER_NAMES[category]]
    folder = next((c for c in candidates if os.path.isdir(c)), None)
    if folder is None:
        print("Folder not found for", category, "->", candidates)
        continue
    for filename in sorted(os.listdir(folder)):
        if filename.lower().endswith(EXTENSIONS):
            records.append({"path": os.path.join(folder, filename),
                            "category": category,
                            "label": LABEL_MAP[category]})

file_df = pd.DataFrame(records)

if SIMULATE_RATIO:   # testing only
    n_min = int(round((file_df.label == 0).sum() / SIMULATE_RATIO))
    minority = file_df[file_df.label == 1].sample(n=n_min, random_state=SEED)
    file_df = pd.concat([file_df[file_df.label == 0], minority]).reset_index(drop=True)

class_counts = file_df.category.value_counts().reindex(CATEGORIES)
for category, count in class_counts.items():
    print(f"{category}: {count} images")

imbalance_ratio = class_counts["normal"] / class_counts["cataract"]
print(f"\nImbalance ratio (normal : cataract) = {imbalance_ratio:.2f} : 1")
print(f"Minority class share = {100 * class_counts['cataract'] / class_counts.sum():.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
bars = ax.bar(class_counts.index, class_counts.values, color=["#4c72b0", "#c44e52"])
for bar, v in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.5, str(v), ha="center")
ax.set_ylabel("Number of images")
ax.set_title("Class Distribution")
plt.tight_layout()
show_fig("class_distribution")

### Think About It
A classifier that predicts *normal* for every image achieves an accuracy equal to the majority-class share. Accuracy alone therefore cannot reveal whether cataract images are detected.

## 4. Inspect Image Sizes Before Resizing

Only the image header is read here, so the check is fast even for high-resolution photographs. Systematic differences in size between classes can act as a shortcut that a model learns instead of the disease.

In [ ]:
for category in CATEGORIES:
    sizes = [Image.open(p).size for p in file_df[file_df.category == category].path]
    print(category, "-> (width, height) counts:", Counter(sizes).most_common(3))

## 5. Preprocessing Class: Crop, Resize and Contrast Enhancement

1. **Crop** the black border around the fundus disc.
2. **Resize** to `IMG_SIZE x IMG_SIZE`, which removes the native-resolution difference between classes.
3. **CLAHE** (contrast-limited adaptive histogram equalisation) on the luminance channel to improve local contrast.

In [ ]:
class FundusPreprocessor:
    """Prepares a fundus photograph for feature extraction and modelling."""

    def __init__(self, size=224, clip_limit=2.0, tile=8):
        self.size = size
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile, tile))

    @staticmethod
    def crop_fundus(bgr, threshold=10):
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        ys, xs = np.where(gray > threshold)
        if len(ys) == 0:
            return bgr
        return bgr[ys.min(): ys.max() + 1, xs.min(): xs.max() + 1]

    def load(self, path):
        """Return the cropped and resized RGB image."""
        bgr = cv2.imread(path)
        if bgr is None:
            return None
        bgr = self.crop_fundus(bgr)
        bgr = cv2.resize(bgr, (self.size, self.size), interpolation=cv2.INTER_AREA)
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    def enhance(self, rgb):
        """Apply CLAHE to the L channel of the LAB colour space.""" # applying image processing methods to enhance the image quality and contrast, which can help in better feature extraction for classification tasks.
        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
        lab[..., 0] = self.clahe.apply(lab[..., 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


preprocessor = FundusPreprocessor(size=IMG_SIZE)

## 6. Load, Preprocess and Label All Images

In [ ]:
images_raw, images, labels, kept_paths = [], [], [], []

for path, label in zip(file_df.path, file_df.label):
    rgb = preprocessor.load(path)
    if rgb is None:
        print("Skipped:", path)
        continue
    images_raw.append(rgb)
    images.append(preprocessor.enhance(rgb))
    labels.append(label)
    kept_paths.append(path)

X_raw = np.array(images_raw)       # cropped + resized
X_images = np.array(images)        # cropped + resized + CLAHE
y = np.array(labels)

print("X_images:", X_images.shape)
print("y:", y.shape)
print("Normal  :", np.sum(y == 0))
print("Cataract:", np.sum(y == 1))

In [ ]:
n = 4
fig, axes = plt.subplots(3, n, figsize=(3 * n, 9))
normal_idx = np.where(y == 0)[0][:n]
cataract_idx = np.where(y == 1)[0][:n]

for col in range(n):
    axes[0, col].imshow(X_images[normal_idx[col]])
    axes[0, col].set_title("Normal")
    axes[1, col].imshow(X_images[cataract_idx[col]])
    axes[1, col].set_title("Cataract")
    axes[2, col].imshow(X_raw[cataract_idx[col]])
    axes[2, col].set_title("Cataract (before CLAHE)")
for ax in axes.flatten():
    ax.axis("off")
plt.tight_layout()
show_fig("sample_images")

## 7. Feature Maps: Canny Edges and Harris Corners

Cataract opacity reduces the visibility of fine retinal vessels, so edge and corner responses are expected to be weaker in cataract images.

In [ ]:
edge_maps, corner_maps, corner_density = [], [], []

for image in X_images:
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    edge_maps.append(cv2.Canny(gray, 50, 120))

    response = cv2.cornerHarris(np.float32(gray), 2, 3, 0.04)
    # Corner density uses the raw response: share of pixels above 1% of the image maximum.
    corner_density.append((response > 0.01 * response.max()).mean())
    # The normalised map is used only for display.
    corner_maps.append(cv2.normalize(response, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8))

edge_maps = np.array(edge_maps)
corner_maps = np.array(corner_maps)
corner_density = np.array(corner_density)
print("Edge maps  :", edge_maps.shape)
print("Corner maps:", corner_maps.shape)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(7, 10))
for col, (idx, name) in enumerate([(normal_idx[0], "Normal"), (cataract_idx[0], "Cataract")]):
    axes[0, col].imshow(X_images[idx]);                  axes[0, col].set_title(f"{name}: Original")
    axes[1, col].imshow(edge_maps[idx], cmap="gray");    axes[1, col].set_title(f"{name}: Canny")
    axes[2, col].imshow(corner_maps[idx], cmap="gray");  axes[2, col].set_title(f"{name}: Harris")
for ax in axes.flatten():
    ax.axis("off")
plt.tight_layout()
show_fig("feature_maps")

## 8. Feature Vectors for the Classical Models

| Group | Descriptor | Length |
|---|---|---|
| Colour | 16-bin histogram per RGB channel | 48 |
| Texture | GLCM contrast, correlation, energy, homogeneity (2 distances x 2 angles) | 16 |
| Texture | Uniform LBP histogram | 10 |
| Shape | HOG on the green channel (128 x 128) | 324 |
| Clarity | Canny edge density, Harris corner density (raw response above 1% of its maximum), Laplacian variance | 3 |

In [ ]:
def colour_histogram(image, bins=16):
    feats = []
    for channel in range(3):
        hist = cv2.calcHist([image], [channel], None, [bins], [0, 256]).flatten()
        feats.extend(hist / (hist.sum() + 1e-8))
    return feats


def glcm_features(gray, levels=32):
    quantised = (gray // (256 // levels)).astype(np.uint8)
    glcm = graycomatrix(quantised, distances=[1, 3], angles=[0, np.pi / 2],
                        levels=levels, symmetric=True, normed=True)
    props = ["contrast", "correlation", "energy", "homogeneity"]
    return np.concatenate([graycoprops(glcm, p).ravel() for p in props])


def lbp_histogram(gray, P=8, R=1):
    lbp = local_binary_pattern(gray, P, R, method="uniform")
    hist, _ = np.histogram(lbp, bins=P + 2, range=(0, P + 2), density=True)
    return hist


def hog_features(gray):
    small = cv2.resize(gray, (128, 128), interpolation=cv2.INTER_AREA)
    return hog(small, orientations=9, pixels_per_cell=(32, 32), cells_per_block=(2, 2))


def extract_feature_vector(image, edge_map, corner_dens):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    green = image[..., 1]
    clarity = [edge_map.mean() / 255.0,
               corner_dens,
               cv2.Laplacian(gray, cv2.CV_64F).var()]
    return np.concatenate([colour_histogram(image), glcm_features(green),
                           lbp_histogram(green), hog_features(green), clarity])


X_features = np.array([extract_feature_vector(img, e, c)
                       for img, e, c in zip(X_images, edge_maps, corner_density)], dtype=np.float32)

print("Combined X_features:", X_features.shape)
print("Labels:", y.shape)

In [ ]:
from scipy.stats import mannwhitneyu

clarity_df = pd.DataFrame(X_features[:, -3:],
                          columns=["Edge density", "Corner density", "Laplacian variance"])
clarity_df["Class"] = np.where(y == 0, "Normal", "Cataract")
print(clarity_df.groupby("Class").agg(["mean", "std"]).round(4))

# Mann-Whitney U test: do the clarity features differ between the classes?
for col in ["Edge density", "Corner density", "Laplacian variance"]:
    p_value = mannwhitneyu(clarity_df.loc[y == 0, col], clarity_df.loc[y == 1, col]).pvalue
    print(f"{col}: p = {p_value:.2e}")

## 9. Stratified Train/Test Split

A stratified split preserves the original imbalance in both partitions. The test set is never resampled, so it reflects the real class distribution.

In [ ]:
idx_all = np.arange(len(y))
idx_train, idx_test = train_test_split(idx_all, test_size=0.20, random_state=SEED, stratify=y)

X_train_f, X_test_f = X_features[idx_train], X_features[idx_test]
y_train, y_test = y[idx_train], y[idx_test]

print("Train:", X_train_f.shape, "| cataract =", np.sum(y_train == 1), "| normal =", np.sum(y_train == 0))
print("Test :", X_test_f.shape, "| cataract =", np.sum(y_test == 1), "| normal =", np.sum(y_test == 0))

## 10. Imbalance-Handling and Evaluation Classes

Three techniques are compared with a no-handling baseline:

| Technique | Level | Idea |
|---|---|---|
| Class weights | Algorithm | Misclassifying the minority class costs more |
| Random oversampling (ROS) | Data | Duplicate minority samples |
| SMOTE | Data | Interpolate new minority samples between nearest neighbours |
| Augmentation oversampling | Data (images) | Rotated and flipped minority images (CNN only) |

Resampling is always applied to **training data only**.

In [ ]:
class ImbalanceHandler:
    """Rebalances a training set so that the minority class matches the majority size."""

    def __init__(self, k_neighbors=5, seed=SEED):
        self.k = k_neighbors
        self.rng = np.random.default_rng(seed)

    def random_oversample(self, X, y):
        minority = np.argmin(np.bincount(y))
        X_min = X[y == minority]
        n_new = np.bincount(y).max() - len(X_min)
        new = X_min[self.rng.integers(0, len(X_min), n_new)]
        return np.vstack([X, new]), np.concatenate([y, np.full(n_new, minority)])

    def smote(self, X, y):
        minority = np.argmin(np.bincount(y))
        X_min = X[y == minority]
        n_new = np.bincount(y).max() - len(X_min)
        k = min(self.k, len(X_min) - 1)
        neighbours = NearestNeighbors(n_neighbors=k + 1).fit(X_min).kneighbors(
            X_min, return_distance=False)[:, 1:]
        base = self.rng.integers(0, len(X_min), n_new)
        partner = neighbours[base, self.rng.integers(0, k, n_new)]
        gap = self.rng.random((n_new, 1))
        new = X_min[base] + gap * (X_min[partner] - X_min[base])
        return np.vstack([X, new]), np.concatenate([y, np.full(n_new, minority)])

    def resample(self, X, y, strategy):
        if strategy == "ros":
            return self.random_oversample(X, y)
        if strategy == "smote":
            return self.smote(X, y)
        return X, y


class ModelEvaluator:
    """Computes metrics beyond accuracy and stores the results of every experiment."""

    def __init__(self):
        self.rows, self.curves = [], {}

    @staticmethod
    def metrics(y_true, y_pred, score):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        return {"Accuracy": (tp + tn) / (tp + tn + fp + fn),
                "Balanced Acc": balanced_accuracy_score(y_true, y_pred),
                "Precision": precision_score(y_true, y_pred, zero_division=0),
                "Recall": recall_score(y_true, y_pred, zero_division=0),
                "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
                "F1": f1_score(y_true, y_pred, zero_division=0),
                "MCC": matthews_corrcoef(y_true, y_pred),
                "ROC-AUC": roc_auc_score(y_true, score),
                "PR-AUC": average_precision_score(y_true, score),
                "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

    def add(self, family, model, strategy, y_true, y_pred, score, **extra):
        row = {"Family": family, "Model": model, "Strategy": strategy}
        row.update(self.metrics(y_true, y_pred, score))
        row.update(extra)
        self.rows.append(row)
        self.curves[(model, strategy)] = (y_true, score)

    def table(self):
        return pd.DataFrame(self.rows)

    def plot_curves(self, keys, title):
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
        for key in keys:
            y_true, score = self.curves[key]
            fpr, tpr, _ = roc_curve(y_true, score)
            prec, rec, _ = precision_recall_curve(y_true, score)
            ax[0].plot(fpr, tpr, label=f"{key[0]} + {key[1]}")
            ax[1].plot(rec, prec, label=f"{key[0]} + {key[1]}")
        ax[0].plot([0, 1], [0, 1], "k--", lw=0.8)
        ax[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curve")
        ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curve")
        ax[1].legend(fontsize=7)
        fig.suptitle(title)
        plt.tight_layout()
        show_fig(title)

    def plot_confusion(self, keys, title):
        fig, axes = plt.subplots(1, len(keys), figsize=(3.4 * len(keys), 3.4))
        axes = np.atleast_1d(axes)
        for ax, key in zip(axes, keys):
            row = next(r for r in self.rows if (r["Model"], r["Strategy"]) == key)
            cm = np.array([[row["TN"], row["FP"]], [row["FN"], row["TP"]]])
            ax.imshow(cm, cmap="Blues")
            for (i, j), v in np.ndenumerate(cm):
                ax.text(j, i, v, ha="center", va="center", fontsize=13)
            ax.set_xticks([0, 1], ["Normal", "Cataract"])
            ax.set_yticks([0, 1], ["Normal", "Cataract"])
            ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
            ax.set_title(f"{key[0]}\n{key[1]}", fontsize=9)
        fig.suptitle(title)
        plt.tight_layout()
        show_fig(title)


evaluator = ModelEvaluator()

### Six-model extension (additional comparison experiments)

- **Classical:** SVM (RBF), Random Forest, **Logistic Regression (L2)**. Each uses standardisation, training-only PCA, optional training-only ROS/SMOTE, and grid-search F1 on training folds.
- **Neural:** Small CNN, MobileNetV2, **EfficientNetB0**. Each uses the same 224×224 RGB inputs, the same stratified train/validation/test split and the same three strategies (none, class weights, minority augmentation).
- **Important input-scale distinction:** SmallCNN uses `[0,1]`, MobileNetV2 uses `[-1,1]`, EfficientNetB0 uses `[0,255]` because rescaling is built into the pretrained model.
- **Expected experiments:** 3 classical × 4 strategies + 3 neural × 3 strategies = **21**, evaluated once on the same held-out test images.
- **Do not claim improvement before running and validating the new results.** Existing 14-model results belong to the prior notebook; rerunning may introduce small variation, especially for neural networks.
- Section 15 saves **SVM + SMOTE**, a selected demonstration model, *not automatically the highest-MCC model* after new experiments. Choose a deployment model only after reviewing the complete comparison.

See `RUN_6_MODELS_README.md` for usage and report update instructions.

## 11. Classical Models: SVM and Random Forest

For every model and strategy:

1. Standardise features and reduce them with PCA (95% variance). Both are fitted on training data only.
2. Resample the training data (ROS or SMOTE) if required.
3. Tune hyper-parameters with 5-fold stratified cross-validation, maximising the **F1-score of the cataract class**. Resampling occurs inside each fold, so synthetic samples never leak into validation folds.
4. Refit on the full training set and evaluate once on the untouched test set.

In [ ]:
MODEL_GRIDS = {
    "SVM": {"C": [1, 10, 100], "gamma": ["scale", 0.01, 0.001]},
    "RandomForest": {"n_estimators": [200, 400], "max_depth": [None, 10]},
    "LogisticRegression": {"C": [0.01, 0.1, 1.0, 10.0]},
}
STRATEGIES = ["none", "class_weight", "ros", "smote"]


def fit_predict(model_name, params, strategy, X_tr, y_tr, X_ev):
    """Fit one pipeline on (X_tr, y_tr) and return predictions and scores for X_ev."""
    scaler = StandardScaler().fit(X_tr)
    pca = PCA(n_components=0.95, random_state=SEED).fit(scaler.transform(X_tr))
    transform = lambda A: pca.transform(scaler.transform(A))

    X_res, y_res = ImbalanceHandler().resample(transform(X_tr), y_tr, strategy)
    weight = "balanced" if strategy == "class_weight" else None

    if model_name == "SVM":
        clf = SVC(kernel="rbf", class_weight=weight, random_state=SEED, **params)
    elif model_name == "RandomForest":
        clf = RandomForestClassifier(class_weight=weight, random_state=SEED, n_jobs=-1, **params)
    elif model_name == "LogisticRegression":
        # L2-regularised linear baseline; class weighting applies only for the corresponding strategy.
        clf = LogisticRegression(solver="lbfgs", max_iter=2000,
                                 class_weight=weight, random_state=SEED, **params)
    else:
        raise ValueError(f"Unknown classical model: {model_name}")
    clf.fit(X_res, y_res)

    X_ev_t = transform(X_ev)
    score = (clf.decision_function(X_ev_t) if model_name == "SVM"
             else clf.predict_proba(X_ev_t)[:, 1])
    return clf.predict(X_ev_t), score


def tune(model_name, strategy, n_splits=5):
    """Grid search with stratified CV; resampling is applied inside every training fold."""
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    best_params, best_f1 = None, -1
    for params in ParameterGrid(MODEL_GRIDS[model_name]):
        fold_f1 = []
        for tr, va in cv.split(X_train_f, y_train):
            pred, _ = fit_predict(model_name, params, strategy, X_train_f[tr], y_train[tr], X_train_f[va])
            fold_f1.append(f1_score(y_train[va], pred, zero_division=0))
        if np.mean(fold_f1) > best_f1:
            best_params, best_f1 = params, float(np.mean(fold_f1))
    return best_params, best_f1


for model_name in MODEL_GRIDS:
    for strategy in STRATEGIES:
        params, cv_f1 = tune(model_name, strategy)
        pred, score = fit_predict(model_name, params, strategy, X_train_f, y_train, X_test_f)
        evaluator.add("Classical", model_name, strategy, y_test, pred, score,
                      **{"CV F1": cv_f1, "Params": str(params)})
        row = evaluator.rows[-1]
        print(f"{model_name:12s} {strategy:12s} CV-F1={cv_f1:.3f} | test F1={row['F1']:.3f} "
              f"recall={row['Recall']:.3f} spec={row['Specificity']:.3f} AUC={row['ROC-AUC']:.3f}")

In [ ]:
classical_df = evaluator.table()
display_cols = ["Model", "Strategy", "Accuracy", "Balanced Acc", "Precision", "Recall",
                "Specificity", "F1", "MCC", "ROC-AUC", "PR-AUC"]
classical_df[display_cols].round(3)

In [ ]:
classical_keys = [(r["Model"], r["Strategy"]) for r in evaluator.rows if r["Family"] == "Classical"]
evaluator.plot_curves(classical_keys, "Classical Models: Test Set")
evaluator.plot_confusion([k for k in classical_keys if k[0] == "SVM"], "SVM: Confusion Matrices")

### 11.1 Robustness Check: Repeated Stratified Cross-Validation

The test set holds only 20 cataract images, so one image changes the recall by 5 percentage points. The classical pipelines are therefore also evaluated with 5-fold stratified cross-validation repeated 3 times on all 400 images, using the hyper-parameters selected in Section 11. Resampling is applied inside each training fold. The reported values are means and standard deviations over the 15 folds.

In [ ]:
import ast
from sklearn.model_selection import RepeatedStratifiedKFold

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
cv_rows = []

for row in [r for r in evaluator.rows if r["Family"] == "Classical"]:
    params = ast.literal_eval(row["Params"])
    for tr, te in rskf.split(X_features, y):
        pred, score = fit_predict(row["Model"], params, row["Strategy"],
                                  X_features[tr], y[tr], X_features[te])
        m = ModelEvaluator.metrics(y[te], pred, score)
        cv_rows.append({"Model": row["Model"], "Strategy": row["Strategy"],
                        **{k: m[k] for k in ["Recall", "Specificity", "Precision", "F1", "MCC", "ROC-AUC"]}})

cv_df = pd.DataFrame(cv_rows)
cv_summary = cv_df.groupby(["Model", "Strategy"]).agg(["mean", "std"]).round(3)
cv_summary.columns = [f"{a} ({b})" for a, b in cv_summary.columns]
cv_summary.to_csv("repeated_cv_results_6_models.csv")
cv_summary

## 12. Neural Architectures: Small CNN, MobileNetV2 and EfficientNetB0 (Transfer Learning)

The CNN receives the preprocessed RGB images. The same `idx_train` and `idx_test` indices are used, so both model families are evaluated on identical test images. A validation subset (20% of training data, stratified) is used for early stopping.

Three imbalance strategies are compared per neural architecture: no handling, class weights, and **augmentation-based oversampling** (rotated and flipped copies of minority images, as in Tutorial 2).

In [ ]:
def rotate_image(image, angle):
    h, w = image.shape[:2]
    matrix = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(image, matrix, (w, h))


def augment_minority(X, y, seed=SEED):
    """Create rotated / flipped copies of minority images until the classes are balanced."""
    rng = np.random.default_rng(seed)
    counts = np.bincount(y)
    minority = np.argmin(counts)
    X_min = X[y == minority]
    new_images = []
    for _ in range(counts.max() - len(X_min)):
        img = X_min[rng.integers(0, len(X_min))]
        img = rotate_image(img, rng.uniform(-25, 25))
        if rng.random() < 0.5:
            img = cv2.flip(img, 1)
        new_images.append(img)
    return (np.concatenate([X, np.array(new_images)]),
            np.concatenate([y, np.full(len(new_images), minority)]))


def build_small_cnn():
    model = Sequential([
        Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        Conv2D(16, (3, 3), activation="relu"), MaxPooling2D((2, 2)),
        Conv2D(32, (3, 3), activation="relu"), MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation="relu"), MaxPooling2D((2, 2)),
        GlobalAveragePooling2D(),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid")])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model


def build_mobilenet():
    base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False                       # feature extractor is frozen
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = GlobalAveragePooling2D()(base(inputs, training=False))
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation="sigmoid")(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model


def build_efficientnet():
    # ImageNet pretraining, freeze convolutional base to limit overfitting on 400 images.
    # EfficientNetB0 includes its own preprocessing and expects float32 pixels in [0, 255].
    base = EfficientNetB0(weights="imagenet", include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = GlobalAveragePooling2D()(base(inputs, training=False))
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation="sigmoid")(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model


def scale_input(X, model_name):
    X = X.astype("float32")
    if model_name == "MobileNetV2":
        return preprocess_input(X)       # rescales [0,255] to [-1,1]
    if model_name == "EfficientNetB0":
        return X                          # EfficientNetB0 rescales internally
    return X / 255.0                      # Small CNN works on [0,1]


In [ ]:
cnn_histories = {}

if tf is not None:
    X_cnn_train, X_cnn_test = X_images[idx_train], X_images[idx_test]
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_cnn_train, y_train, test_size=0.20, random_state=SEED, stratify=y_train)

    for model_name, builder in [
        ("SmallCNN", build_small_cnn),
        ("MobileNetV2", build_mobilenet),
        ("EfficientNetB0", build_efficientnet),
    ]:
        for strategy in ["none", "class_weight", "augment_oversample"]:
            tf.keras.backend.clear_session()
            tf.random.set_seed(SEED)

            X_fit, y_fit = (augment_minority(X_tr, y_tr) if strategy == "augment_oversample"
                            else (X_tr, y_tr))
            weights = None
            if strategy == "class_weight":
                w = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_fit)
                weights = {0: w[0], 1: w[1]}

            model = builder()
            history = model.fit(
                scale_input(X_fit, model_name), y_fit,
                validation_data=(scale_input(X_val, model_name), y_val),
                epochs=30, batch_size=16, class_weight=weights, verbose=0,
                callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)])
            cnn_histories[(model_name, strategy)] = history.history

            score = model.predict(scale_input(X_cnn_test, model_name), verbose=0).ravel()
            pred = (score >= 0.5).astype(int)
            evaluator.add("Neural", model_name, strategy, y_test, pred, score,
                          **{"Epochs": len(history.history["loss"])})
            row = evaluator.rows[-1]

            # Persist the exact candidate from this experiment (including its
            # restored best-validation weights), rather than rebuilding a new model later.
            if model_name == "EfficientNetB0" and strategy == "class_weight":
                model.save("cataract_efficientnetb0_class_weight.keras")
                print("Saved: cataract_efficientnetb0_class_weight.keras")

            print(f"{model_name:12s} {strategy:18s} epochs={row['Epochs']:2d} | test F1={row['F1']:.3f} "
                  f"recall={row['Recall']:.3f} spec={row['Specificity']:.3f} AUC={row['ROC-AUC']:.3f}")
else:
    print("TensorFlow is required for this section.")

In [ ]:
# Diagnostic: score distribution of the Small CNN trained with augmentation oversampling
key = ("SmallCNN", "augment_oversample")

if key in evaluator.curves:
    y_true, scores = evaluator.curves[key]
    print("Minimum score               :", round(float(scores.min()), 4))
    print("Maximum score               :", round(float(scores.max()), 4))
    print("Mean score                  :", round(float(scores.mean()), 4))
    print("Mean score, normal images   :", round(float(scores[y_true == 0].mean()), 4))
    print("Mean score, cataract images :", round(float(scores[y_true == 1].mean()), 4))
    print("Images predicted as cataract at threshold 0.5:", int(np.sum(scores >= 0.5)))
else:
    print("Run Section 12 first.")


## 13. Compare All Models

In [ ]:

# SECTION 13: FINAL MODEL COMPARISON

results_df = evaluator.table().copy()

ranking = results_df.sort_values(
    ["MCC", "F1"],
    ascending=[False, False]
).reset_index(drop=True)

display_cols = [
    "Family", "Model", "Strategy",
    "Accuracy", "Recall", "Specificity",
    "Precision", "F1", "MCC",
    "ROC-AUC", "PR-AUC", "FN"
]

print("Total experiments:", len(ranking), "/ 21")
if len(ranking) != 21:
    print("WARNING: Expected 21 results (12 classical + 9 neural). Check that all cells completed.")

# Display Accuracy as percentage
display(
    ranking[display_cols].style.format({
        "Accuracy": "{:.1%}",
        "Recall": "{:.1%}",
        "Specificity": "{:.1%}",
        "Precision": "{:.1%}",
        "F1": "{:.3f}",
        "MCC": "{:.3f}",
        "ROC-AUC": "{:.3f}",
        "PR-AUC": "{:.3f}"
    })
)

# Save original numerical values for analysis
ranking.to_csv("model_comparison_6_models.csv", index=False)


In [ ]:
if len(results_df):
    labels_plot = results_df.Model + "\n" + results_df.Strategy
    x = np.arange(len(results_df))
    fig, ax = plt.subplots(figsize=(max(10, 0.7 * len(results_df)), 4.5))
    ax.bar(x - 0.27, results_df["Recall"], 0.27, label="Recall (sensitivity)")
    ax.bar(x, results_df["Specificity"], 0.27, label="Specificity")
    ax.bar(x + 0.27, results_df["F1"], 0.27, label="F1 (cataract)")
    ax.set_xticks(x, labels_plot, rotation=60, ha="right", fontsize=7)
    ax.set_ylim(0, 1.05)
    ax.set_title("Performance on the Minority and Majority Classes")
    ax.legend(fontsize=8)
    plt.tight_layout()
    show_fig("performance_bars")

In [ ]:
if tf is not None and cnn_histories:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for key, h in cnn_histories.items():
        axes[0].plot(h["loss"], label=f"{key[0]} + {key[1]}")
        axes[1].plot(h["val_loss"], label=f"{key[0]} + {key[1]}")
    axes[0].set(title="Training Loss", xlabel="Epoch")
    axes[1].set(title="Validation Loss", xlabel="Epoch")
    axes[1].legend(fontsize=6)
    plt.tight_layout()
    show_fig("training_curves")

    neural_keys = [(r["Model"], r["Strategy"]) for r in evaluator.rows if r["Family"] == "Neural"]
    evaluator.plot_curves(neural_keys, "Neural Models: Test Set")
    evaluator.plot_confusion(neural_keys, "Neural Models: Confusion Matrices")

## 14. Select the Best Combination

The final model is chosen by the **Matthews correlation coefficient (MCC)** and the **F1-score of the cataract class**, because both remain informative under imbalance. Recall is inspected next, since a missed cataract case is the costlier error.

In [ ]:
best = results_df.sort_values(["MCC", "F1"], ascending=False).iloc[0]
print("Best combination:", best["Model"], "+", best["Strategy"])
metrics = ["Accuracy", "Balanced Acc", "Precision", "Recall",
           "Specificity", "F1", "MCC", "ROC-AUC"]
print(best[metrics].astype(float).round(3))

results_df.to_csv("final_results_6_models.csv", index=False)
print("\nSaved: final_results_6_models.csv")

## 14A. Visual Summary of the Results

These charts are generated from `results_df` (Section 13), so they always match the latest run. They support the Results and Discussion section of the report and the dashboard. Run Sections 1 to 14 first. After the full run, save the notebook with its outputs (or use File, Export As, HTML) so the charts stay visible to a reader who does not re-run the code.

In [ ]:
# Shared labels and colours for the summary charts
STRAT_LABEL = {"none": "baseline", "class_weight": "class weights", "ros": "ROS",
               "smote": "SMOTE", "augment_oversample": "augmentation"}
FAMILY_COLOUR = {"Classical": "#4C72B0", "Neural": "#DD8452"}

summary = results_df.copy()
summary["Config"] = summary["Model"] + " + " + summary["Strategy"].map(STRAT_LABEL).fillna(summary["Strategy"])
summary["Colour"] = summary["Family"].map(FAMILY_COLOUR)
print("Experiments in summary:", len(summary))

In [ ]:
# Chart 1: ranking of all experiments by MCC (the primary selection criterion)
d = summary.sort_values("MCC")
fig, ax = plt.subplots(figsize=(8, 0.32 * len(d) + 1.5))
ax.barh(d["Config"], d["MCC"], color=d["Colour"])
for i, v in enumerate(d["MCC"]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8)
ax.set_xlabel("Matthews correlation coefficient (test set)")
ax.set_title("All experiments ranked by MCC (blue = classical, orange = neural)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
show_fig("chart_mcc_ranking")

In [ ]:
# Chart 2: recall versus specificity (the cost of catching more cataract images)
fig, ax = plt.subplots(figsize=(7, 5))
markers = {"none": "o", "class_weight": "s", "ros": "^", "smote": "D", "augment_oversample": "P"}
for strat, g in summary.groupby("Strategy"):
    ax.scatter(g["Specificity"], g["Recall"], c=g["Colour"], marker=markers.get(strat, "o"),
               s=60, edgecolor="white", label=STRAT_LABEL.get(strat, strat))
best_row = summary.sort_values(["MCC", "F1"], ascending=False).iloc[0]
ax.annotate(best_row["Config"], (best_row["Specificity"], best_row["Recall"]),
            xytext=(0.02, 0.05), textcoords="axes fraction", fontsize=8,
            arrowprops=dict(arrowstyle="-", color="grey"))
ax.set(xlabel="Specificity (normal images)", ylabel="Recall (cataract images)",
       title="Recall versus specificity")
ax.legend(title="Strategy (marker)", fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
show_fig("chart_recall_vs_specificity")

In [ ]:
# Chart 3: accuracy paradox. Accuracy near 0.75 can hide a recall of 0
fig, ax = plt.subplots(figsize=(7, 5))
for fam, g in summary.groupby("Family"):
    ax.scatter(g["Accuracy"], g["Recall"], c=FAMILY_COLOUR[fam], label=fam, s=55, edgecolor="white")
ax.axvline(0.75, ls="--", c="grey")
ax.text(0.752, 0.02, "all-normal baseline (75% accuracy)", fontsize=8, color="dimgrey")
ax.set(xlabel="Accuracy", ylabel="Cataract recall", title="Accuracy hides missed cataract images")
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout()
show_fig("chart_accuracy_paradox")

In [ ]:
# Chart 4: effect of class weighting against the baseline for every model
pivot = summary.pivot_table(index="Model", columns="Strategy", values=["Recall", "MCC"])
effect = pd.DataFrame({
    "Recall baseline": pivot["Recall"]["none"], "Recall class weights": pivot["Recall"]["class_weight"],
    "MCC baseline": pivot["MCC"]["none"], "MCC class weights": pivot["MCC"]["class_weight"]}).round(3)
display(effect)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
effect[["Recall baseline", "Recall class weights"]].plot.bar(ax=axes[0], color=["#999999", "#C44E52"])
effect[["MCC baseline", "MCC class weights"]].plot.bar(ax=axes[1], color=["#999999", "#4C72B0"])
axes[0].set_title("Recall: baseline versus class weights")
axes[1].set_title("MCC: baseline versus class weights")
for a in axes:
    a.set_xlabel(""); a.tick_params(axis="x", rotation=30); a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
show_fig("chart_class_weight_effect")

In [ ]:
# Chart 5: confusion matrix of the selected best configuration
best_key = (best_row["Model"], best_row["Strategy"])
evaluator.plot_confusion([best_key], f"Best configuration: {best_row['Config']}")

In [ ]:
# Chart 6: clarity features by class (supports the diagnostic discussion)
clarity_cols = list(clarity_df.columns[:3])
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, col in zip(axes, clarity_cols):
    data = [clarity_df.loc[clarity_df["Class"] == c, col] for c in ["Normal", "Cataract"]]
    ax.boxplot(data, patch_artist=True,
               boxprops=dict(facecolor="#DDE6F2"), medianprops=dict(color="#C44E52"))
    ax.set_xticks([1, 2], ["Normal", "Cataract"]); ax.set_title(col, fontsize=10); ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Clarity measures by class")
plt.tight_layout()
show_fig("chart_clarity_by_class")

In [ ]:
# Uncertainty of recall on only 20 cataract test images (95% Wilson interval)
def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return centre - half, centre + half

top = summary.sort_values(["MCC", "F1"], ascending=False).head(5)[["Config", "TP", "FN", "MCC"]].copy()
top["Recall 95% CI"] = [f"{lo:.2f} to {hi:.2f}" for lo, hi in
                        (wilson(int(r.TP), int(r.TP + r.FN)) for r in top.itertuples())]
display(top)
print("Overlapping intervals mean the 80-image test set cannot separate the top configurations.")

## 15. Predict a New Fundus Image with the Classical Model (SVM + SMOTE)

Section 15A fits the scaler, PCA and SVM on the training split only, then stores them as one bundle. Section 15B applies the identical preprocessing and 401-feature extraction to a new image.


In [ ]:

# SECTION 15A: SAVE SELECTED MODEL
# Run after completing Sections 1-14

import ast
import joblib

# Get the selected hyperparameters from Section 11
selected_row = results_df[
    (results_df["Model"] == "SVM") &
    (results_df["Strategy"] == "smote")
].iloc[0]

params = ast.literal_eval(selected_row["Params"])

# Fit scaling and PCA using training data ONLY
final_scaler = StandardScaler()
X_scaled = final_scaler.fit_transform(X_train_f)

final_pca = PCA(n_components=0.95, random_state=SEED)
X_pca = final_pca.fit_transform(X_scaled)

# SMOTE only on training features
handler = ImbalanceHandler(seed=SEED)
X_balanced, y_balanced = handler.resample(
    X_pca, y_train, "smote"
)

# Train selected SVM
final_model = SVC(
    kernel="rbf",
    random_state=SEED,
    **params
)

final_model.fit(X_balanced, y_balanced)

# Save entire inference pipeline
joblib.dump({
    "scaler": final_scaler,
    "pca": final_pca,
    "classifier": final_model
}, "cataract_svm_smote.joblib")

print("Model successfully saved!")


In [ ]:

# SECTION 15B: PREDICT NEW IMAGE

import joblib

bundle = joblib.load("cataract_svm_smote.joblib")

def predict_new_image(image_path):

    # 1. Preprocess image
    rgb = preprocessor.load(image_path)

    if rgb is None:
        raise ValueError("Unable to read image")

    enhanced = preprocessor.enhance(rgb)

    # 2. Extract clarity measurements
    gray = cv2.cvtColor(
        enhanced, cv2.COLOR_RGB2GRAY
    )

    edges = cv2.Canny(gray, 50, 120)

    response = cv2.cornerHarris(
        np.float32(gray), 2, 3, 0.04
    )

    corner_dens = (
        response > 0.01 * response.max()
    ).mean()

    # 3. Extract all 401 features
    features = extract_feature_vector(
        enhanced, edges, corner_dens
    )

    features = np.asarray(
        features, dtype=np.float32
    ).reshape(1, -1)

    # 4. Apply saved scaler and PCA
    scaled = bundle["scaler"].transform(features)
    reduced = bundle["pca"].transform(scaled)

    # 5. Make prediction
    prediction = int(
        bundle["classifier"].predict(reduced)[0]
    )

    label = "Cataract" if prediction == 1 else "Normal"

    print("Predicted Class:", label)
    return prediction


# Set this to the path of a fundus photograph that is not part of the training data.
NEW_IMAGE_PATH = "new_image.png"

if os.path.isfile(NEW_IMAGE_PATH):
    predict_new_image(NEW_IMAGE_PATH)
else:
    print("Set NEW_IMAGE_PATH to an existing fundus image to run the demonstration.")


## 16. Predict a New Fundus Image with EfficientNetB0 + Class Weights

Section 12 saves the trained EfficientNetB0 + class-weight model as `cataract_efficientnetb0_class_weight.keras`. Preprocessing of a new image must match Section 5: black-border crop, 224 × 224 resizing and LAB/CLAHE. EfficientNetB0 receives float32 pixels in the range 0 to 255 and must not be divided by 255, because the Keras application includes its own input scaling.

The output is a research demonstration of a two-class classifier, not a clinical diagnosis. The printed sigmoid score is not a calibrated disease probability, and the input must be a fundus photograph.


In [ ]:
# SECTION 16: USE THE SAVED EFFICIENTNETB0 MODEL FOR A NEW IMAGE
from pathlib import Path

MODEL_PATH = "cataract_efficientnetb0_class_weight.keras"

def predict_efficientnet_image(image_path, model_path=MODEL_PATH):
    if tf is None:
        raise RuntimeError("TensorFlow is required for EfficientNetB0 inference")
    if not Path(model_path).is_file():
        raise FileNotFoundError(f"Run Section 12 first to save the trained model: {model_path}")

    rgb = preprocessor.load(str(image_path))
    if rgb is None:
        raise ValueError(f"Unreadable or missing fundus image: {image_path}")
    enhanced = preprocessor.enhance(rgb)
    X_new = scale_input(enhanced[None, ...], "EfficientNetB0")

    saved_model = tf.keras.models.load_model(model_path)
    score = float(saved_model.predict(X_new, verbose=0)[0, 0])
    label = "Cataract" if score >= 0.5 else "Normal"
    print(f"Prediction: {label} | sigmoid score: {score:.4f} (uncalibrated)")
    return {"label": label, "score": score}

# Set this to the path of a fundus photograph that is not part of the training data.
NEW_FUNDUS_PATH = "new_image.png"
if Path(NEW_FUNDUS_PATH).is_file():
    predict_efficientnet_image(NEW_FUNDUS_PATH)
else:
    print("Set NEW_FUNDUS_PATH to an existing fundus image to run a demo.")


## 17. Export Data for the Streamlit Dashboard

The dashboard reads four small CSV files so that every chart in the web application is built from the same results as this notebook. Run this section once, after Sections 1 to 14A and Section 11.1. The files are written straight into `../assets/data/` of the dashboard repository, so no copying is needed. Commit them to GitHub together with the code.

In [ ]:
# SECTION 17: EXPORT DATA FOR THE STREAMLIT DASHBOARD
from pathlib import Path

REPO_DATA = Path("..") / "assets" / "data"          # notebook sits in "Cataract Classification/"
EXPORT_DIR = REPO_DATA if REPO_DATA.is_dir() else Path("dashboard_export")
EXPORT_DIR.mkdir(exist_ok=True)

STRATEGY_NAMES = {"none": "baseline", "class_weight": "class weights", "ros": "ROS",
                  "smote": "SMOTE", "augment_oversample": "augmentation"}
MODEL_NAMES = {"RandomForest": "Random Forest", "LogisticRegression": "Logistic Regression",
               "SmallCNN": "Small CNN"}

def tidy_model(name):
    return MODEL_NAMES.get(name, name)

# 1. results.csv: one row per experiment (21 rows)
tbl = evaluator.table()
results_export = pd.DataFrame({
    "family": tbl["Family"], "model": tbl["Model"].map(tidy_model),
    "strategy": tbl["Strategy"].map(STRATEGY_NAMES),
    "TP": tbl["TP"], "FN": tbl["FN"], "TN": tbl["TN"], "FP": tbl["FP"],
    "accuracy": tbl["Accuracy"], "recall": tbl["Recall"], "specificity": tbl["Specificity"],
    "precision": tbl["Precision"], "f1": tbl["F1"], "mcc": tbl["MCC"],
    "roc_auc": tbl["ROC-AUC"], "avg_precision": tbl["PR-AUC"],
}).round(3)
results_export.to_csv(EXPORT_DIR / "results.csv", index=False)

# 2. repeated_cv.csv: mean and SD over 5-fold x 3 repeats (Section 11.1)
cv_export = cv_summary.reset_index()
cv_export = pd.DataFrame({
    "model": cv_export["Model"].map(tidy_model),
    "strategy": cv_export["Strategy"].map(STRATEGY_NAMES),
    "recall_mean": cv_export["Recall (mean)"], "recall_sd": cv_export["Recall (std)"],
    "f1_mean": cv_export["F1 (mean)"], "f1_sd": cv_export["F1 (std)"],
    "mcc_mean": cv_export["MCC (mean)"], "mcc_sd": cv_export["MCC (std)"],
})
cv_export.to_csv(EXPORT_DIR / "repeated_cv.csv", index=False)

# 3. clarity_features.csv: the three clarity measurements of all 400 images (Section 6)
clarity_export = pd.DataFrame(X_features[:, -3:],
                              columns=["edge_density", "corner_density", "laplacian_variance"])
clarity_export["class"] = np.where(y == 0, "Normal", "Cataract")
clarity_export.round(5).to_csv(EXPORT_DIR / "clarity_features.csv", index=False)

# 4. test_scores.csv: label and sigmoid score of every test image for each neural experiment
score_rows = []
for r in evaluator.rows:
    if r["Family"] != "Neural":
        continue
    y_true, scores = evaluator.curves[(r["Model"], r["Strategy"])]
    for yt, sc in zip(np.asarray(y_true), np.asarray(scores).ravel()):
        score_rows.append({"model": tidy_model(r["Model"]), "strategy": STRATEGY_NAMES[r["Strategy"]],
                           "y_true": int(yt), "score": round(float(sc), 5)})
pd.DataFrame(score_rows).to_csv(EXPORT_DIR / "test_scores.csv", index=False)

for f in sorted(EXPORT_DIR.glob("*.csv")):
    print(f"{f.name:24s} {len(pd.read_csv(f)):5d} rows")
print("Saved to:", EXPORT_DIR.resolve())